# Lesson 09 — Calculus Intuition

**What this notebook does:** builds a one-knob urgency scorer for support tickets, measures how wrong it is with a *loss*, measures the loss's *slope* by nudging the knob (no formulas needed), then walks downhill to the best setting with *gradient descent*. Along the way it plots the walk, deliberately breaks it with a too-big learning rate, and finishes with two knobs at once — which is where the *gradient* appears.

Run the cells top to bottom.

## One knob to tune

Four support tickets, described by a single signal each: how many urgent-sounding words they contain (`urgent_words`). A human has already assigned each one a "true" urgency score (`true_urgency`) — this tiny made-up dataset is our ground truth.

Our scorer has exactly **one knob**: a weight `w`. Its prediction for every ticket is simply `w × urgent_words` — Lesson 08's weighted-signals idea, cut down to one signal so we can watch the knob clearly.

To tune the knob we first need a way to measure *how wrong* a given setting is. That measuring stick is the **loss**: for each ticket, take `(prediction − truth)`, **square it** (so overshooting and undershooting both count as wrong, and big misses count extra), then **average** over all tickets. One number out: `0` means a perfect scorer, bigger means worse.

The cell below tries four knob settings by hand. Watch which one wins.

In [ ]:
import numpy as np

urgent_words = np.array([0, 1, 2, 4])
true_urgency = np.array([0, 2, 4, 8])

def predict(w):
    return w * urgent_words

def loss(w):
    return ((predict(w) - true_urgency) ** 2).mean()

for guess in [0.0, 1.0, 2.0, 3.0]:
    print(f"w = {guess}: loss = {loss(guess)}")

## Measuring the slope by nudging

Guessing knob settings one at a time is blind. The smarter question to ask at any current setting is: *"if I turn the knob a tiny bit to the right, does the loss go up or down — and how fast?"*

That number is the **slope** of the loss at that point (its formal name is the *derivative*). And we can measure it with no formulas at all:

1. Nudge `w` a hair **up** and compute the loss there.
2. Nudge `w` the same hair **down** and compute the loss there.
3. Subtract, and divide by the distance between the two nudged points.

That is literally "rise over run" from school, measured over a very small run (we use `0.0001`). Read the result like this:

- **Sign** = direction: a *negative* slope means the loss falls as `w` grows (downhill is to the right); a *positive* slope means downhill is to the left.
- **Size** = steepness: a slope of `-21` means the loss is falling steeply; `-0.01` means the ground is nearly flat.

Check the printed values against the previous cell: at `w = 0` the loss is falling fast toward the right (we saw 21.0 drop to 5.25), and at `w = 3` it rises to the right — both signs will match what the numbers already showed.

In [ ]:
def slope(f, w, nudge=0.0001):
    return (f(w + nudge) - f(w - nudge)) / (2 * nudge)

for point in [0.0, 1.0, 3.0]:
    print(f"slope of the loss at w = {point}: {slope(loss, point):.1f}")

## Walking downhill: gradient descent

Now use the slope to *improve* the knob automatically. The recipe:

1. Measure the slope at the current `w`.
2. Take a small step in the **opposite** direction — slope negative means loss falls to the right, so step right; slope positive, step left. In code both cases collapse into one line: `w = w - learning_rate * slope`.
3. Repeat.

The **learning rate** is the step size — how big a fraction of the slope we actually move. We use `0.05` here.

This recipe is called **gradient descent**, and it is *the* algorithm by which almost every model in this course learns, from Phase 2's regressions to the largest language models.

Watch the printout: `w` climbs from `0` toward `2`, the loss collapses toward `0`, and the steps shrink on their own — because the valley gets flatter near the bottom, the measured slope shrinks, so `learning_rate × slope` shrinks too. Nobody schedules that; it falls out of the math.

In [ ]:
w = 0.0
learning_rate = 0.05
path = [w]

for step in range(1, 21):
    w = w - learning_rate * slope(loss, w)
    path.append(w)
    if step <= 5 or step % 5 == 0:
        print(f"step {step:2d}: w = {w:.3f}, loss = {loss(w):.3f}")

## Seeing the walk

One picture makes the whole lesson click. Compute the loss for every value of `w` between `-0.5` and `4.5` and plot it — you get a **valley** with its bottom exactly at `w = 2`. Then mark every stop from our descent on top of it.

Two things to look for in the picture:

- The red dots start high on the left wall and slide down to the bottom.
- The dots are **far apart at first and bunch up near the bottom** — nobody told the walk to slow down; the flattening slope did that automatically.

The figure is also saved as a PNG in a `plots/` folder next to this notebook (same pattern as Lesson 07).

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

Path("plots").mkdir(exist_ok=True)

ws = np.linspace(-0.5, 4.5, 200)
curve = [loss(v) for v in ws]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ws, curve, color="#2a78d6", linewidth=2, label="loss for every possible w")
ax.plot(path, [loss(v) for v in path], "o--", color="#e34948",
        markersize=6, linewidth=1.5, label="our descent, step by step")
ax.set_xlabel("weight w")
ax.set_ylabel("loss (how wrong the scorer is)")
ax.set_title("Gradient descent walking down the loss valley")
ax.grid(color="#e1e0d9", linewidth=0.8)
ax.legend()
fig.savefig("plots/lesson-09-gradient-descent.png", dpi=150, bbox_inches="tight")
plt.show()

## When the step is too big

The learning rate is a real choice, and here is what happens when it is chosen badly. With `0.2` instead of `0.05`, every step *overshoots* the bottom of the valley and lands higher up the opposite wall. From there the slope is even steeper, so the next step overshoots even more.

Watch `w` bounce from one side of `2.0` to the other while the loss **grows** every step instead of shrinking. This is called *diverging*, and it is one of the most common ways training a real model fails.

In [ ]:
w = 0.0
too_big = 0.2

for step in range(1, 7):
    w = w - too_big * slope(loss, w)
    print(f"step {step}: w = {w:.3f}, loss = {loss(w):.3f}")

## Two knobs at once: the gradient

Real models have many knobs, not one. To see what changes, give the scorer a second knob: a **baseline** `b` added to every prediction, so `prediction = w × urgent_words + b`. The new targets below follow the pattern `2 × urgent_words + 1`, so the best setting is `w = 2, b = 1` — but the code does not know that; it has to find it.

With two knobs, "the slope" becomes **two slopes**: nudge only `w` to ask "does loss drop if I turn the weight knob?", then nudge only `b` to ask the same about the baseline knob. Each is called a *partial* slope because it nudges one knob while holding the other still.

The list of all the slopes, one per knob, is the **gradient** — and a list of numbers with fixed positions is exactly a *vector* from Lesson 08. The gradient vector points in the direction that makes the loss grow fastest, so stepping *against* it is the fastest way down.

The recipe does not change at all: measure every slope, step every knob opposite its own slope, repeat. That, scaled up, is how every neural network learns.

In [ ]:
true_urgency_2 = np.array([1, 3, 5, 9])

def loss2(w, b):
    return ((w * urgent_words + b - true_urgency_2) ** 2).mean()

def slope_w(w, b, nudge=0.0001):
    return (loss2(w + nudge, b) - loss2(w - nudge, b)) / (2 * nudge)

def slope_b(w, b, nudge=0.0001):
    return (loss2(w, b + nudge) - loss2(w, b - nudge)) / (2 * nudge)

print(f"gradient at w=0, b=0: [{slope_w(0, 0):.1f}, {slope_b(0, 0):.1f}]")

w, b = 0.0, 0.0
print(f"start: loss = {loss2(w, b):.3f}")

for step in range(400):
    w = w - 0.05 * slope_w(w, b)
    b = b - 0.05 * slope_b(w, b)

print(f"after 400 steps: w = {w:.3f}, b = {b:.3f}, loss = {loss2(w, b):.3f}")

## What this means for real models

Everything a neural network does during training is this exact game, scaled up:

- A real model has **millions (or billions) of knobs** instead of two. They are called parameters or weights.
- Training is: compute the loss on real data, get the gradient (one slope per knob), step every knob a little bit opposite its slope, repeat — often millions of times.
- Nudging each knob one at a time, like we did here, would be far too slow for millions of knobs. **Backpropagation** (Phase 3) is a clever bookkeeping method that computes every slope in a single pass. The *idea* is exactly what you just did — only the bookkeeping is faster.

Notice also that two knobs needed 400 steps where one knob needed 20. Knobs interact, and the valley gets trickier to walk as the count grows — that is why Phase 3 spends a whole lesson on making training work well.